In [20]:
import pandas as pd
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
import string

In [21]:
def preprocess(text):
    if pd.isna(text):
        return []
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in string.punctuation]
    return tokens

What data is relevant for the task of finding similar episodes based on descriptions?
What data is relevant for finding an episode suitable for user query?

The episode descriptions from the dw_imdb dataset are relevant for both tasks, as they provide textual information about each episode that can be used to compute similarity. They are also more useful since we'll be only working with the series reboot starting 2005 and not the "old" Doctor Who.

The scripts from the dw_scripts dataset may also be relevant if we want to analyze the content of the episodes in more detail, but for a basic similarity comparison, the descriptions should suffice.

In [22]:
import json

# Открыть JSON-файл и загрузить в словарь
with open("dw_data/document_corpus_dw.json", "r", encoding="utf-8") as f:
    document_corpus = json.load(f)

with open("dw_data/inverted_index.json", "r", encoding="utf-8") as f:
    inverted_index = json.load(f)

In [23]:
def boolean_search(query: str, index: dict):
    query_tokens = preprocess(query)
    if not query_tokens:
      return []
    results = set(index.get(query_tokens[0], []))

    for token in query_tokens[1:]:
      results = results.union(index.get(token, []))

    return list(results)[:5]

In [24]:
from rank_bm25 import BM25Okapi

# Подготовка корпуса для BM25
# Сохраняем ключи в том же порядке, в котором будут токенизированные описания
doc_keys = list(document_corpus.keys())
tokenized_corpus = [preprocess(document_corpus[k]['description']) for k in doc_keys]

# Создаём BM25 объект
bm25 = BM25Okapi(tokenized_corpus)

# Функция поиска
def bm25_search(query, n=5):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)
    
    # Индексы топ-n документов
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    results = []
    for i in ranked_indices[:n]:
        doc_id = doc_keys[i]  # берём ключ из согласованного списка
        results.append({
            'episodeid': doc_id,
            'title': document_corpus[doc_id]['title'],
            'score': scores[i],
            'snippet': document_corpus[doc_id]['description'][:300]  # первые 300 символов
        })
    return results

In [25]:
from rank_bm25 import BM25Okapi

# Prepare corpus for BM25
tokenized_corpus = [preprocess(doc['description']) for doc in document_corpus.values()]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, n=5):
    query_tokens = preprocess(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    results = []
    for i in ranked_indices[:n]:
        doc_id = list(document_corpus.keys())[i]
        results.append({
            'episodeid': doc_id,
            'title': document_corpus[doc_id]['title'],
            'score': scores[i],
            'snippet': document_corpus[doc_id]['description'][:300]  # first 300 chars of episode
        })
    return results

In [26]:
import json

queries = [
    "Doctor fights with Weeping Angels and wants to save Amy",
    "Doctor and Clara are in nineteenth century",
    "Doctor meet River Song for the first time",
    "Doctor and Donna encounter Daleks and Davros",
    "Doctor and Rose end up in a parallel universe",
    "Doctor meets van Gogh",
    "Doctor and Martha face time paradox creatures or similar threats",
    "Doctor and Bill encounter Cybermen",
    "Paternoster Gang and Doctor and Vastra and Strax",
    "Doctor and Rose meet her father Pete Tyler"
]

answers = [
    ["5x4", "5x5", "7x5", "6x11", "3x10"],
    ["7x6", "7x12", "7x8", "1x3", "7x10"],
    ["4x8", "4x9", "5x4", "5x5", "6x1"],
    ["4x12", "4x13", "2x12", "2x13", "9x1"],
    ["2x5", "2x6", "2x12", "2x13", "4x11"],
    ["5x10", "5x1", "5x12", "5x13", "1x3"],
    ["3x10", "3x8", "3x9", "3x11", "3x12"],
    ["10x11", "10x12", "2x5", "2x6", "8x12"],
    ["7x6", "7x11", "7x13", "8x1", "6x7"],
    ["1x8", "2x5", "2x6", "1x13", "4x11"]
]

data = {"queries": queries, "answers": answers}

with open("dw_data/first_example_10_queries.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

In [27]:
BOLD = "\033[1m"
RESET = "\033[0m"

for i, query in enumerate(queries):
    print(f"\n{BOLD}Query '{query}'{RESET}")
    hits = boolean_search(query, inverted_index)
    relevant_docs = answers[i]
    boolean_correct_count = 0
    bm_25_correct_count = 0

    # Keep only hits present in document_corpus
    hits = [id for id in hits if id in document_corpus]
    
    print(f"\nBoolean hits:")
    for i, id in enumerate(hits):
        episode = document_corpus.get(id)
        print(f"{i+1}: {id} - {episode['title']}")
        if res['episodeid'] in relevant_docs:
            boolean_correct_count += 1

    print(f"{boolean_correct_count}/{5} documents were a right guess")
    
    print("\nBM25 Results:")
    results = bm25_search(query, n=5)
    for i, res in enumerate(results):
        print(f"Rank {i+1}: {res['episodeid']} - {res['title']}")
        if res['episodeid'] in relevant_docs:
            bm_25_correct_count += 1

    print(f"{bm_25_correct_count}/{5} documents were a right guess")


Query 'Doctor fights with Weeping Angels and wants to save Amy'

Boolean hits:
1: 9x6 - The Woman Who Lived
2: 7x12 - The Crimson Horror
3: 8x1 - Deep Breath
4: 8x2 - Into the Dalek
5: 10x7 - The Pyramid at the End of the World
0/5 documents were a right guess

BM25 Results:
Rank 1: 7x5 - The Angels Take Manhattan
Rank 2: 5x4 - The Time of Angels
Rank 3: 5x5 - Flesh and Stone
Rank 4: 6x11 - The God Complex
Rank 5: 6x0 - A Christmas Carol
4/5 documents were a right guess

Query 'Doctor and Clara are in nineteenth century'

Boolean hits:
1: 9x6 - The Woman Who Lived
2: 7x12 - The Crimson Horror
3: 8x1 - Deep Breath
4: 8x2 - Into the Dalek
5: 10x7 - The Pyramid at the End of the World
0/5 documents were a right guess

BM25 Results:
Rank 1: 5x6 - The Vampires of Venice
Rank 2: 2x2 - Tooth and Claw
Rank 3: 7x12 - The Crimson Horror
Rank 4: 2x4 - The Girl in the Fireplace
Rank 5: 4x3 - Planet of the Ood
1/5 documents were a right guess

Query 'Doctor meet River Song for the first time'

Boo

In [28]:
results_table = []

for i, query in enumerate(queries):
    relevant_docs = set(answers[i])

    # Boolean search
    hits_boolean = [doc_id for doc_id in boolean_search(query, inverted_index) if doc_id in document_corpus][:5]
    boolean_correct = sum(1 for doc_id in hits_boolean if doc_id in relevant_docs)

    # BM25 search
    bm25_results = bm25_search(query, n=5)
    bm25_top5 = [res['episodeid'] for res in bm25_results]
    bm25_correct = sum(1 for doc_id in bm25_top5 if doc_id in relevant_docs)

    # Добавляем строку в таблицу
    results_table.append({
        "Query": query,
        "Boolean Correct": boolean_correct,
        "BM25 Correct": bm25_correct
    })

# Создаём DataFrame
df_results = pd.DataFrame(results_table)

# Сохраняем в CSV (или Excel)
df_results.to_csv("dw_data/search_results_summary.csv", index=False)

# Показать кратко в Jupyter
print(df_results)

                                               Query  Boolean Correct  \
0  Doctor fights with Weeping Angels and wants to...                0   
1         Doctor and Clara are in nineteenth century                1   
2          Doctor meet River Song for the first time                0   
3       Doctor and Donna encounter Daleks and Davros                0   
4      Doctor and Rose end up in a parallel universe                0   
5                              Doctor meets van Gogh                0   
6  Doctor and Martha face time paradox creatures ...                0   
7                 Doctor and Bill encounter Cybermen                0   
8   Paternoster Gang and Doctor and Vastra and Strax                1   
9         Doctor and Rose meet her father Pete Tyler                0   

   BM25 Correct  
0             4  
1             1  
2             2  
3             2  
4             1  
5             1  
6             0  
7             1  
8             0  
9             1 